# Bank Account Fraud Detection · Risk-Based Investigation

**Portfolio notebook · Ilia Jakhaia**  
BAF NeurIPS 2022 (`Base.csv`) · chronological evaluation · capacity-aware fraud detection.

> This notebook is a reproducible rewrite of the submitted analysis. The GitHub README displays results recorded in the original executed notebook; they may vary slightly across software versions. This notebook has no pre-populated outputs and will only show results after execution on your local copy of `Base.csv`.

## 1. Setup

From the repository root run `pip install -e ".[notebook]"`. Download `Base.csv` into `data/`, or set `CSV_PATH` below. No user-specific home directory is hardcoded.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from baf_fraud.data import load_base, chronological_split
from baf_fraud.modeling import build_models
from baf_fraud.evaluation import evaluate_model
from baf_fraud.plotting import plot_distributions, plot_validation, plot_test

CSV_PATH = Path("data/Base.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError("Download BAF Base.csv into data/Base.csv (instructions in data/README.md)")
df = load_base(CSV_PATH)
print(f"Rows: {len(df):,} | columns: {df.shape[1]}")
display(df.head(3))

## 2. Exploratory analysis

Class imbalance is extreme, so overall accuracy alone is not a useful success measure. The source benchmark uses month indices **0–7**, not named calendar months. Note that feature-level missing states may be encoded as negative sentinels.

In [ ]:
print(df["fraud_bool"].value_counts().rename(index={0: "Legitimate", 1: "Fraud"}))
print(f"Overall fraud prevalence: {df['fraud_bool'].mean():.3%}")
display(df.groupby("month")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean"))
display(df.groupby("foreign_request")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean"))
plot_distributions(df, "results/generated/figures")

## 3. Chronological holdout

- Train: months 0–5
- Validation and model selection: month 6
- Final untouched test: month 7

Both `fraud_bool` and `month` are removed from model input. Preprocessors are fitted inside the training pipelines.

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = chronological_split(df)
for name, y in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:>11}: {len(y):>8,} | prevalence: {y.mean():.3%}")

## 4. Train two reproducible baselines

Logistic regression uses `class_weight='balanced'`; random forest uses `class_weight='balanced_subsample'`. We evaluate Average Precision and a **1% manual-review capacity** on validation.

In [ ]:
models = build_models(X_train)
validation = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    validation[name] = evaluate_model(y_val, model.predict_proba(X_val)[:, 1])
validation_table = pd.DataFrame(validation).T
cols = ["average_precision", "precision_at_capacity", "recall_at_capacity", "frauds_detected"]
display(validation_table[cols])
plot_validation(validation, "results/generated/figures")

## 5. Select on validation **before** scoring the test set

Select highest **Average Precision**, using capacity precision as tie-breaker. This avoids model selection on test data.

In [ ]:
selected_name = max(sorted(validation), key=lambda name: (
    validation[name]["average_precision"],
    validation[name]["precision_at_capacity"]
))
print("Selected model:", selected_name)

## 6. One-pass test evaluation

This section should be run **once** after finalizing selection. We report both 0.5-threshold classification metrics and the review-at-capacity operating point. These are different operating policies.

In [ ]:
test_probs = models[selected_name].predict_proba(X_test)[:, 1]
test = evaluate_model(y_test, test_probs, capacity=.01)
display(pd.Series(test, name="Held-out test metrics"))
plot_test(y_test.to_numpy(), test_probs, "results/generated/figures")
print(f"At 1% review capacity: {test['frauds_detected']} detected / {test['reviewed']} reviewed")

## 7. Interpret results responsibly

- **Precision@1%** is the fraud share among top-risk applications reviewed.
- **Recall@1%** is the fraction of all fraud captured at fixed capacity.
- **Average Precision** summarizes ranking across thresholds, not precision at 0.5.
- Drift and false positives motivate calibration, monitoring, and fairness/error analysis.
- This is a **synthetic-data research benchmark**, not a production banking decision system.